# Naija Airline Price Finder ✈️💸
This notebook uses a custom Playwright scraper (`scraper.py`) to bypass Javascript rendering and find exact prices on **Travelwings** using **OpenAI Function Calling**.
Just enter the origin, destination, and date, and we'll dynamically build the URL to scrape the results and return them in Pidgin!

In [2]:
import os 
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from scraper import scrape_travelwings_prices

ModuleNotFoundError: No module named 'playwright'

In [ ]:
# Load environment variables (Make sure your OPENAI_API_KEY is active!)
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("Ah ah! No API key was found - please make sure it's set in your environment variables!")
else:
    print("API key found and we are good to go!")

client = OpenAI(api_key=api_key)

In [ ]:
# System prompt instructing our AI
system_prompt = """
You are the "Naija Ticket Master" — a very sharp Nigerian travel agent.

Your job is to read the user request, figure out the Origin, Destination, and Travel Date, and then USE YOUR `check_travelwings` TOOL.
Note: The tool requires 3-letter IATA airport codes (e.g., Abuja = ABV, Lagos = LOS) and dates in YYYY-MM-DD format.

Your Rules:
1. Speak completely in fun, relatable Nigerian Pidgin English.
2. Always call the `check_travelwings` tool when a user asks for a flight.
3. Wait for the tool output, which will give you the scraped text from Travelwings website.
4. Read the scraped text carefully. Look for airlines (e.g., Air Peace, APG Airlines) and their exact prices (e.g., NGN 92,611).
5. List out the available airlines and their prices using bullet points. Mention if they are Economy, Business etc.
6. If no prices are found, say so.
7. End with a funny one-liner about traveling in Nigeria.
"""

# Define the tool schema for OpenAI
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_travelwings",
            "description": "Scrapes Travelwings flight data for a specific route and returns unstructured text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {
                        "type": "string",
                        "description": "The 3-letter IATA code for the departure city (e.g., ABV for Abuja)."
                    },
                    "destination": {
                        "type": "string",
                        "description": "The 3-letter IATA code for the arrival city (e.g., LOS for Lagos)."
                    },
                    "date": {
                        "type": "string",
                        "description": "The date of travel in YYYY-MM-DD format."
                    }
                },
                "required": ["origin", "destination", "date"],
                "additionalProperties": False,
            },
        }
    }
]

def handle_chat(user_message: str, history) -> str:
    messages = [{"role": "system", "content": system_prompt}]
    
    # Append history
    for past_user, past_assistant in history:
        messages.append({"role": "user", "content": past_user})
        messages.append({"role": "assistant", "content": past_assistant})
        
    messages.append({"role": "user", "content": user_message})

    try:
        # Loop to handle potential multiple rounds of tool calls
        while True:
            # 1. Ask OpenAI, giving it access to our tool
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools,
                temperature=0.7
            )
            
            response_message = response.choices[0].message
            
            # 2. Check if the model wants to call our tool
            if response_message.tool_calls:
                messages.append(response_message)
                
                for tool_call in response_message.tool_calls:
                    if tool_call.function.name == "check_travelwings":
                        args = json.loads(tool_call.function.arguments)
                        print(f"Executing Playwright Scraper for: {args}")
                        
                        origin = args.get("origin")
                        destination = args.get("destination")
                        date_val = args.get("date")
                        
                        # Run Playwright scraper (Can take ~10-15 seconds)
                        function_result = scrape_travelwings_prices(origin, destination, date_val)
                        
                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": function_result
                        })
                # Loop continues to send tool results back to the model
            else:
                # No more tool calls, return the final response
                return response_message.content
                
    except Exception as e:
        return f"Omo, wahala dey! Error don occur: {str(e)}"


In [ ]:
# Set up the Gradio App using ChatInterface for back-and-forth conversation
demo = gr.ChatInterface(
    fn=handle_chat,
    title="Naija Airline Ticket Master ✈️💸",
    description="Just tell me where you wanna go and when! (e.g. 'Abuja to Lagos on 2026-03-03')",
    examples=[
        "Check flight prices from Abuja to Lagos on 2026-03-03",
        "Any Kano to Abuja flights for next week?"
    ]
)

# Launch with sharing turned ON so you can run it and open on your phone or send to a friend!
demo.launch(share=True)